# Im et al. CSCW 2018 — Dataset Reproduction

Reproduces and extends the dataset from:
> Im et al. "Deliberation and Resolution on Wikipedia: A Case Study of Requests for Comment" CSCW 2018

**Their method**: track Legobot's edits on talk pages.
- Legobot adds an RFC ID when `{{rfc}}` is placed → marks RfC open
- Legobot removes the tag with "Removing expired RFC template" → marks RfC stale
- Non-Legobot removal of the tag → formally or informally closed

**Our extension**: 2011–2025 (vs. their 2011–2017)

**Outputs** (all in `~/rfc-analysis/data/imetal/`):
- `rfc_opens.csv` — all Legobot RFC ID additions (one row per RfC open event)
- `rfc_closes.csv` — all close events matched to opens
- `rfc_manifest.csv` — one row per RfC: page, open/close timestamps, outcome type, closer
- `rfc_participation.csv` — editors per RfC between open and close
- `rfc_closers.csv` — closer experience stats

**Note on comment table**: Wikimedia replicas store edit summaries in a separate
`comment` table joined via `rev_comment_id`. If that join fails, fall back to
`rev_comment` direct column (older replica schema).

In [ ]:
import wmpaws
import pandas as pd
from pathlib import Path

OUT = Path.home() / 'rfc-analysis' / 'data' / 'imetal'
OUT.mkdir(parents=True, exist_ok=True)
WIKI = 'enwiki'
print(f'Output folder: {OUT}')

## Step 0 — Verify Legobot actor ID and comment table schema

First confirm Legobot's actor_id and check whether the `comment` table
is available or whether edit summaries are in `rev_comment` directly.

In [ ]:
df_legobot = wmpaws.run_sql("""
SELECT actor_id, actor_user, actor_name
FROM actor
WHERE actor_name = 'Legobot'
""", WIKI)
print(df_legobot)
LEGOBOT_ACTOR_ID = int(df_legobot['actor_id'].iloc[0])
print(f'Legobot actor_id: {LEGOBOT_ACTOR_ID}')

In [ ]:
# Check whether comment table exists and has the right columns
try:
    df_test = wmpaws.run_sql("""
    SELECT r.rev_id, c.comment_text
    FROM revision r
    JOIN comment c ON r.rev_comment_id = c.comment_id
    WHERE r.rev_actor = {actor_id}
    LIMIT 3
    """.format(actor_id=LEGOBOT_ACTOR_ID), WIKI)
    USE_COMMENT_TABLE = True
    print('comment table available')
    print(df_test)
except Exception as e:
    USE_COMMENT_TABLE = False
    print(f'comment table unavailable ({e}), will use rev_comment directly')

In [ ]:
# If comment table not available, check rev_comment column
if not USE_COMMENT_TABLE:
    df_test2 = wmpaws.run_sql("""
    SELECT rev_id, rev_comment
    FROM revision
    WHERE rev_actor = {actor_id}
    LIMIT 3
    """.format(actor_id=LEGOBOT_ACTOR_ID), WIKI)
    print(df_test2)

## Step 1 — RfC openings

Every Legobot edit with summary containing "Adding RFC ID" on a talk page
marks the moment an RfC was registered. One row = one RfC open event.

Talk pages span multiple namespaces (1=Talk, 3=User talk, 5=Wikipedia talk,
etc.) — we collect all of them so we can filter later.

In [ ]:
if USE_COMMENT_TABLE:
    opens_sql = """
    SELECT
        r.rev_id                        AS open_rev_id,
        r.rev_page                      AS page_id,
        p.page_namespace                AS namespace,
        p.page_title                    AS page_title,
        r.rev_timestamp                 AS open_ts
    FROM revision r
    JOIN page p    ON r.rev_page = p.page_id
    JOIN comment c ON r.rev_comment_id = c.comment_id
    WHERE r.rev_actor = {actor_id}
      AND c.comment_text LIKE '%Adding RFC ID%'
      AND r.rev_timestamp >= '20110101000000'
    ORDER BY r.rev_timestamp
    """.format(actor_id=LEGOBOT_ACTOR_ID)
else:
    opens_sql = """
    SELECT
        r.rev_id                        AS open_rev_id,
        r.rev_page                      AS page_id,
        p.page_namespace                AS namespace,
        p.page_title                    AS page_title,
        r.rev_timestamp                 AS open_ts
    FROM revision r
    JOIN page p ON r.rev_page = p.page_id
    WHERE r.rev_actor = {actor_id}
      AND r.rev_comment LIKE '%Adding RFC ID%'
      AND r.rev_timestamp >= '20110101000000'
    ORDER BY r.rev_timestamp
    """.format(actor_id=LEGOBOT_ACTOR_ID)

df_opens = wmpaws.run_sql(opens_sql, WIKI)
df_opens.to_csv(OUT / 'rfc_opens.csv', index=False)
print(f'{len(df_opens):,} RfC open events')
print(df_opens['namespace'].value_counts())
df_opens.head()

## Step 2 — Stale closes (Legobot)

Legobot removes the expired RFC template when no closer has acted within
the default window (~30 days). These are the "stale" RfCs — dispute unresolved.

In [ ]:
if USE_COMMENT_TABLE:
    stale_sql = """
    SELECT
        r.rev_id                        AS close_rev_id,
        r.rev_page                      AS page_id,
        r.rev_timestamp                 AS close_ts,
        'stale'                         AS outcome_type,
        'Legobot'                       AS closer
    FROM revision r
    JOIN comment c ON r.rev_comment_id = c.comment_id
    WHERE r.rev_actor = {actor_id}
      AND c.comment_text LIKE '%Removing expired RFC%'
      AND r.rev_timestamp >= '20110101000000'
    ORDER BY r.rev_timestamp
    """.format(actor_id=LEGOBOT_ACTOR_ID)
else:
    stale_sql = """
    SELECT
        r.rev_id                        AS close_rev_id,
        r.rev_page                      AS page_id,
        r.rev_timestamp                 AS close_ts,
        'stale'                         AS outcome_type,
        'Legobot'                       AS closer
    FROM revision r
    WHERE r.rev_actor = {actor_id}
      AND r.rev_comment LIKE '%Removing expired RFC%'
      AND r.rev_timestamp >= '20110101000000'
    ORDER BY r.rev_timestamp
    """.format(actor_id=LEGOBOT_ACTOR_ID)

df_stale = wmpaws.run_sql(stale_sql, WIKI)
df_stale.to_csv(OUT / 'rfc_stale_closes.csv', index=False)
print(f'{len(df_stale):,} stale close events')
df_stale.head()

## Step 3 — Human closes (formal + informal)

When a human (not Legobot) closes an RfC, their edit summary typically contains
"closing rfc", "rfc closed", "close rfc", "rfc result", etc.

We collect all non-Legobot edits on pages that had an RfC open, with
close-signal keywords in the edit summary. We then classify:
- **Formal close**: closer was not the initiator or a participant (requires
  cross-referencing participant list — done in Step 5)
- **Informal close**: closer was initiator or participant

This is an approximation; Im et al. used full wikitext diffs to detect
template removal. We use edit summary heuristics here.

In [ ]:
# Get the set of page_ids that ever had an RfC open
rfc_page_ids = df_opens['page_id'].unique().tolist()
print(f'{len(rfc_page_ids):,} unique pages with RfC opens')

# Query human closes via edit summary keywords
# Process in batches to avoid IN clause size limits
BATCH = 5000
human_close_rows = []

close_keywords = "rfc closed|closing rfc|close rfc|rfc result|request for comment closed|closing request"

for i in range(0, len(rfc_page_ids), BATCH):
    batch = rfc_page_ids[i:i+BATCH]
    ids_str = ','.join(str(x) for x in batch)

    if USE_COMMENT_TABLE:
        sql = f"""
        SELECT
            r.rev_id                    AS close_rev_id,
            r.rev_page                  AS page_id,
            r.rev_timestamp             AS close_ts,
            a.actor_name                AS closer,
            c.comment_text              AS close_comment
        FROM revision r
        JOIN actor a   ON r.rev_actor = a.actor_id
        JOIN comment c ON r.rev_comment_id = c.comment_id
        WHERE r.rev_page IN ({ids_str})
          AND r.rev_actor != {LEGOBOT_ACTOR_ID}
          AND LOWER(c.comment_text) REGEXP 'rfc closed|closing rfc|close rfc|rfc result'
          AND r.rev_timestamp >= '20110101000000'
        ORDER BY r.rev_timestamp
        """
    else:
        sql = f"""
        SELECT
            r.rev_id                    AS close_rev_id,
            r.rev_page                  AS page_id,
            r.rev_timestamp             AS close_ts,
            a.actor_name                AS closer,
            r.rev_comment               AS close_comment
        FROM revision r
        JOIN actor a ON r.rev_actor = a.actor_id
        WHERE r.rev_page IN ({ids_str})
          AND r.rev_actor != {LEGOBOT_ACTOR_ID}
          AND LOWER(r.rev_comment) REGEXP 'rfc closed|closing rfc|close rfc|rfc result'
          AND r.rev_timestamp >= '20110101000000'
        ORDER BY r.rev_timestamp
        """

    batch_df = wmpaws.run_sql(sql, WIKI)
    human_close_rows.append(batch_df)
    print(f'  batch {i//BATCH+1}: {len(batch_df):,} rows')

df_human_closes = pd.concat(human_close_rows, ignore_index=True)
df_human_closes.to_csv(OUT / 'rfc_human_closes.csv', index=False)
print(f'\n{len(df_human_closes):,} human close events')
df_human_closes.head()

## Step 4 — Build RfC manifest

Match each open event to its closest subsequent close event on the same page.
Priority: human close > stale close. If neither found, mark as 'open' (still active
or missed by heuristics).

One page can have multiple RfCs over time — each open is matched independently.

In [ ]:
import numpy as np

# Combine human and stale closes
df_stale['close_comment'] = 'Removing expired RFC template'
df_all_closes = pd.concat([
    df_human_closes[['close_rev_id','page_id','close_ts','closer','close_comment']],
    df_stale[['close_rev_id','page_id','close_ts','closer','close_comment']]
], ignore_index=True).sort_values('close_ts')

# For each open, find the next close on the same page
records = []
for _, row in df_opens.iterrows():
    pid = row['page_id']
    ots = row['open_ts']

    candidates = df_all_closes[
        (df_all_closes['page_id'] == pid) &
        (df_all_closes['close_ts'] > ots)
    ].sort_values('close_ts')

    if len(candidates) == 0:
        records.append({
            **row.to_dict(),
            'close_rev_id': None, 'close_ts': None,
            'outcome_type': 'open', 'closer': None, 'close_comment': None
        })
    else:
        # Prefer human close over stale if both exist nearby
        human = candidates[candidates['closer'] != 'Legobot']
        stale = candidates[candidates['closer'] == 'Legobot']

        if len(human) > 0:
            c = human.iloc[0]
            outcome = 'human_close'
        else:
            c = stale.iloc[0]
            outcome = 'stale'

        records.append({
            **row.to_dict(),
            'close_rev_id': c['close_rev_id'],
            'close_ts': c['close_ts'],
            'outcome_type': outcome,
            'closer': c['closer'],
            'close_comment': c['close_comment']
        })

df_manifest = pd.DataFrame(records)
df_manifest['duration_days'] = df_manifest.apply(
    lambda r: (
        pd.to_datetime(r['close_ts'], format='%Y%m%d%H%M%S') -
        pd.to_datetime(r['open_ts'], format='%Y%m%d%H%M%S')
    ).days if pd.notna(r['close_ts']) else None, axis=1
)

df_manifest.to_csv(OUT / 'rfc_manifest.csv', index=False)
print(f'{len(df_manifest):,} RfCs in manifest')
print(df_manifest['outcome_type'].value_counts())

## Step 5 — Participation per RfC

For each RfC, count distinct editors who edited the talk page between
open and close timestamps. This gives initiator + participant counts.

The **initiator** is the first non-Legobot editor after the open.
The **closer** is already identified in the manifest.
Everyone else is a **participant**.

In [ ]:
# For each RfC in the manifest, get all edits between open and close
# Process in batches grouped by page_id
participation_rows = []

manifest_pages = df_manifest['page_id'].unique().tolist()

for i in range(0, len(manifest_pages), BATCH):
    batch = manifest_pages[i:i+BATCH]
    ids_str = ','.join(str(x) for x in batch)

    sql = f"""
    SELECT
        r.rev_page                      AS page_id,
        r.rev_timestamp                 AS rev_ts,
        a.actor_name                    AS username,
        a.actor_user                    AS user_id,
        u.user_registration             AS registration
    FROM revision r
    JOIN actor a   ON r.rev_actor = a.actor_id
    LEFT JOIN `user` u ON a.actor_user = u.user_id
    LEFT JOIN user_groups ug ON a.actor_user = ug.ug_user AND ug.ug_group = 'bot'
    WHERE r.rev_page IN ({ids_str})
      AND r.rev_actor != {LEGOBOT_ACTOR_ID}
      AND r.rev_timestamp >= '20110101000000'
      AND ug.ug_user IS NULL
    ORDER BY r.rev_page, r.rev_timestamp
    """
    batch_df = wmpaws.run_sql(sql, WIKI)
    participation_rows.append(batch_df)
    print(f'  batch {i//BATCH+1}: {len(batch_df):,} rows')

df_edits = pd.concat(participation_rows, ignore_index=True)

# Now join to manifest windows and aggregate
rfc_participation = []
for _, rfc in df_manifest.iterrows():
    pid = rfc['page_id']
    ots = rfc['open_ts']
    cts = rfc['close_ts'] if pd.notna(rfc['close_ts']) else '99991231000000'

    window = df_edits[
        (df_edits['page_id'] == pid) &
        (df_edits['rev_ts'] >= ots) &
        (df_edits['rev_ts'] <= cts)
    ]

    if len(window) == 0:
        initiator = None
    else:
        initiator = window.sort_values('rev_ts').iloc[0]['username']

    rfc_participation.append({
        'open_rev_id': rfc['open_rev_id'],
        'page_id': pid,
        'page_title': rfc['page_title'],
        'namespace': rfc['namespace'],
        'open_ts': ots,
        'close_ts': rfc['close_ts'],
        'outcome_type': rfc['outcome_type'],
        'closer': rfc['closer'],
        'duration_days': rfc['duration_days'],
        'initiator': initiator,
        'n_participants': window['username'].nunique(),
        'n_edits': len(window),
    })

df_participation = pd.DataFrame(rfc_participation)
df_participation.to_csv(OUT / 'rfc_participation.csv', index=False)
print(f'{len(df_participation):,} RfCs with participation data')
df_participation[['n_participants','n_edits','duration_days']].describe().round(1)

## Step 6 — Closer experience

Im et al. found closers had avg 39,759 edits vs 14,055 for participants.
Reproduce this using total edit counts from the revision table.

In [ ]:
human_closers = df_participation[
    (df_participation['outcome_type'] == 'human_close') &
    df_participation['closer'].notna()
]['closer'].unique().tolist()

print(f'{len(human_closers):,} unique human closers')

if len(human_closers) > 0:
    closers_str = ', '.join(f"'{c.replace("'", "''")}'" for c in human_closers[:500])
    df_closer_exp = wmpaws.run_sql(f"""
    SELECT
        a.actor_name                    AS username,
        COUNT(*)                        AS total_edits,
        MIN(r.rev_timestamp)            AS first_edit,
        MAX(r.rev_timestamp)            AS last_edit
    FROM revision r
    JOIN actor a ON r.rev_actor = a.actor_id
    WHERE a.actor_name IN ({closers_str})
    GROUP BY 1
    ORDER BY total_edits DESC
    """, WIKI)
    df_closer_exp.to_csv(OUT / 'rfc_closers.csv', index=False)
    print(f'Median closer edits: {df_closer_exp["total_edits"].median():,.0f}')
    print(f'Mean closer edits:   {df_closer_exp["total_edits"].mean():,.0f}')
    df_closer_exp.head(10)

## Summary — comparison with Im et al.

In [ ]:
outcome_counts = df_participation['outcome_type'].value_counts()
total = len(df_participation)

formally_closed = outcome_counts.get('human_close', 0)
stale = outcome_counts.get('stale', 0)
still_open = outcome_counts.get('open', 0)

lines = [
    '# Im et al. Reproduction — Summary',
    f'',
    f'Our dataset (2011-present):',
    f'  Total RfCs:            {total:,}',
    f'  Human closed:          {formally_closed:,} ({100*formally_closed/total:.0f}%)',
    f'  Stale (Legobot):       {stale:,} ({100*stale/total:.0f}%)',
    f'  Still open/unclosed:   {still_open:,} ({100*still_open/total:.0f}%)',
    f'',
    f'Im et al. 2011-2017 (reference):',
    f'  Total RfCs:            7,316',
    f'  Formally closed:       4,086 (58%)',
    f'  Informally ended:        672  (9%)',
    f'  Stale:                 2,329 (33%)',
    f'',
    f'Participation (our data):',
    f'  Median participants/RfC: {df_participation["n_participants"].median():.0f}',
    f'  Mean participants/RfC:   {df_participation["n_participants"].mean():.0f}',
    f'  Median duration (days):  {df_participation["duration_days"].median():.1f}',
]

summary = '\n'.join(lines)
print(summary)
(OUT / 'summary.txt').write_text(summary)